In [1]:
"""
Diebold-Mariano Test — Statistical Significance of MSPE Improvements
======================================================================
Input:  Output_All_Forecasts_Combined.xlsx  (same folder as this script)
Output: Output_DM_Test_Results.xlsx

For each model and each horizon h:
    H0: E[d_t] = 0  (equal forecast accuracy vs benchmark)
    d_t = e_model,t^2 - e_benchmark,t^2  (loss differential)

    Two-sided test — tests whether model is better OR worse than benchmark.
    Significance levels following Baumeister et al. (2024):
        ** = significant at 5%  (|t| > critical value at 5%)
        *  = significant at 10% (|t| > critical value at 10%)

    HAC variance: Newey-West with bandwidth = floor(h^(1/3)) per Harvey
    et al. (1997) recommendation for h-step-ahead forecasts.

    Original DM test (Diebold & Mariano 1995) — no small-sample correction.

Reference: Diebold & Mariano (1995), Journal of Business & Economic Statistics
           Baumeister, Huber, Lee & Ravazzolo (2024)
"""

import pandas as pd
import numpy as np
from scipy import stats
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import os
import warnings
warnings.filterwarnings('ignore')

# ── Parameters ────────────────────────────────────────────────────────────────
INPUT_FILE  = 'Output_All_Forecasts_Combined.xlsx'
OUTPUT_FILE = 'Output_DM_Test_Results.xlsx'
BENCHMARK   = 'RW_avg (benchmark)'
HORIZONS    = [1, 3, 6, 9, 12, 15, 18, 21, 24]

script_dir  = os.getcwd()
# Uncomment if running outside Jupyter:
# script_dir = r"C:\Users\yatsk\...\All Models Outputs"

# ── Load data ─────────────────────────────────────────────────────────────────
print("Loading data...")
df = pd.read_excel(os.path.join(script_dir, INPUT_FILE))
df['forecast_origin'] = pd.to_datetime(df['forecast_origin'])
df['forecast_origin'] = df['forecast_origin'] + pd.offsets.MonthEnd(0)
df = df.dropna(subset=['actual'])
df['sq_error'] = (df['forecast'] - df['actual']) ** 2
ALL_MODELS = sorted(df['model'].unique().tolist())
print(f"  Rows: {len(df):,}  |  Models: {len(ALL_MODELS)}  |  "
      f"Origins: {df['forecast_origin'].nunique()}")

# ── Newey-West HAC variance ───────────────────────────────────────────────────
def newey_west_var(d, h):
    """
    Newey-West HAC variance of mean(d).
    Bandwidth = floor(h^(1/3)) — standard for h-step-ahead DM test.
    d: array of loss differentials
    h: forecast horizon
    """
    n   = len(d)
    bw  = int(np.floor(h ** (1/3)))   # bandwidth
    d_c = d - d.mean()                # demeaned

    # Variance = gamma(0) + 2 * sum of weighted autocovariances
    var = np.dot(d_c, d_c) / n
    for j in range(1, bw + 1):
        weight  = 1 - j / (bw + 1)   # Bartlett kernel weight
        gamma_j = np.dot(d_c[j:], d_c[:-j]) / n
        var    += 2 * weight * gamma_j

    return var / n   # variance of the sample mean

# ── DM test ───────────────────────────────────────────────────────────────────
def dm_test(errors_model, errors_bench, h):
    """
    Original Diebold-Mariano (1995) test.
    Two-sided: H0: equal accuracy vs H1: different accuracy.
    Returns: (dm_stat, p_value, stars)
    """
    # Loss differential: positive = model worse than benchmark
    d = errors_model ** 2 - errors_bench ** 2

    n      = len(d)
    d_mean = d.mean()

    # HAC variance
    var_d = newey_west_var(d, h)

    if var_d <= 0:
        return np.nan, np.nan, ''

    dm_stat = d_mean / np.sqrt(var_d)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))   # two-sided

    # Stars following Baumeister et al.
    if p_value <= 0.05:
        stars = '**'
    elif p_value <= 0.10:
        stars = '*'
    else:
        stars = ''

    return dm_stat, p_value, stars

# ── Run DM tests for all models and horizons ──────────────────────────────────
print("Running DM tests...")

results = {}   # {(model, horizon): (mspe_ratio, dm_stat, p_value, stars)}

for h in HORIZONS:
    sub  = df[df['horizon'] == h].copy()
    sub  = sub.sort_values('forecast_origin')

    # Benchmark errors at each origin
    bench_sub = (sub[sub['model'] == BENCHMARK]
                 .set_index('forecast_origin')['forecast']
                 - sub[sub['model'] == BENCHMARK]
                 .set_index('forecast_origin')['actual'])

    bench_mspe = (bench_sub ** 2).mean()

    for m in ALL_MODELS:
        if m == BENCHMARK:
            results[(m, h)] = (1.0, np.nan, np.nan, '')
            continue

        mod_sub = (sub[sub['model'] == m]
                   .set_index('forecast_origin'))

        # Align dates with benchmark
        common  = mod_sub.index.intersection(bench_sub.index)
        if len(common) < 10:
            results[(m, h)] = (np.nan, np.nan, np.nan, '')
            continue

        e_model = mod_sub.loc[common, 'forecast'] - mod_sub.loc[common, 'actual']
        e_bench = bench_sub.loc[common]

        # MSPE ratio
        mspe_ratio = (e_model ** 2).mean() / (e_bench ** 2).mean()

        # DM test
        dm_stat, p_val, stars = dm_test(
            e_model.values, e_bench.values, h)

        results[(m, h)] = (mspe_ratio, dm_stat, p_val, stars)

print("  Done.")

# ── Build summary table ───────────────────────────────────────────────────────
# Wide table: rows = models, columns = horizons
# Each cell: MSPE ratio + stars

rows = []
for m in ALL_MODELS:
    row = {'model': m}
    ratios = []
    for h in HORIZONS:
        ratio, dm, pval, stars = results.get((m, h), (np.nan, np.nan, np.nan, ''))
        row[f'h{h}_ratio']  = ratio
        row[f'h{h}_stars']  = stars
        row[f'h{h}_dm']     = dm
        row[f'h{h}_pval']   = pval
        if not np.isnan(ratio):
            ratios.append(ratio)
    row['avg_ratio'] = np.mean(ratios) if ratios else np.nan
    rows.append(row)

summary = pd.DataFrame(rows).sort_values('avg_ratio')

# Print to terminal
print()
print("=" * 80)
print(f"DM TEST RESULTS — Two-sided, ** = 5%, * = 10%")
print(f"Benchmark: {BENCHMARK}")
print("=" * 80)
header = f"{'Model':<45}" + "".join([f"  h={h:2d}" for h in HORIZONS]) + "  Average"
print(header)
print("-" * 80)
for _, row in summary.iterrows():
    m = row['model']
    line = f"{m:<45}"
    ratios = []
    for h in HORIZONS:
        ratio = row[f'h{h}_ratio']
        stars = row[f'h{h}_stars']
        if np.isnan(ratio):
            line += f"  {'—':>6}"
        else:
            cell = f"{ratio:.3f}{stars}"
            line += f"  {cell:>6}"
            ratios.append(ratio)
    avg = row['avg_ratio']
    line += f"  {avg:.3f}" if not np.isnan(avg) else "  —"
    print(line)

# ── Save to Excel ─────────────────────────────────────────────────────────────
print()
print("Building Excel output...")

BLUE  = PatternFill('solid', fgColor='1F4E79')
DBLUE = PatternFill('solid', fgColor='2E75B6')
LBLUE = PatternFill('solid', fgColor='BDD7EE')
GREEN = PatternFill('solid', fgColor='C6EFCE')
RED   = PatternFill('solid', fgColor='FFC7CE')
GREY  = PatternFill('solid', fgColor='F2F2F2')
WHITE = PatternFill('solid', fgColor='FFFFFF')

def sw(ws, widths):
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w

def c(ws, r, col, v, fill=WHITE, bold=False, fmt=None,
      align='center', color='000000', cs=1):
    if cs > 1:
        ws.merge_cells(start_row=r, start_column=col,
                       end_row=r, end_column=col+cs-1)
    cl = ws.cell(r, col, v)
    cl.font      = Font(bold=bold, size=10, color=color)
    cl.fill      = fill
    cl.alignment = Alignment(horizontal=align, vertical='center',
                              wrap_text=True)
    if fmt:
        cl.number_format = fmt

wb = openpyxl.Workbook()

# ── Sheet 1: MSPE ratios with stars ──────────────────────────────────────────
ws1 = wb.active
ws1.title = 'MSPE Ratios with DM Stars'
sw(ws1, [38] + [10]*9 + [10])

c(ws1, 1, 1,
  f'MSPE Ratios with Diebold-Mariano Significance Stars  |  '
  f'** = 5%  |  * = 10%  |  Two-sided test  |  Benchmark: {BENCHMARK}',
  fill=BLUE, bold=True, color='FFFFFF', cs=11)
ws1.row_dimensions[1].height = 28

c(ws1, 2, 1,
  f'Stars indicate statistical significance of difference from benchmark '
  f'(original DM test, Newey-West HAC, bandwidth = floor(h^(1/3))). '
  f'Green < 0.85  |  Red > 1.15  |  Bold = beats benchmark',
  fill=LBLUE, align='left', cs=11)
ws1.row_dimensions[2].height = 18

c(ws1, 3, 1, 'Model', fill=DBLUE, bold=True, color='FFFFFF', align='left')
for ci, h in enumerate(HORIZONS, 2):
    c(ws1, 3, ci, f'h={h}', fill=DBLUE, bold=True, color='FFFFFF')
c(ws1, 3, 11, 'Average', fill=DBLUE, bold=True, color='FFFFFF')
ws1.row_dimensions[3].height = 20

for ri, (_, row) in enumerate(summary.iterrows()):
    r   = ri + 4
    alt = GREY if ri % 2 == 0 else WHITE
    m   = row['model']
    is_bench = (m == BENCHMARK)

    c(ws1, r, 1, m,
      fill=LBLUE if is_bench else alt, bold=is_bench, align='left')

    for ci, h in enumerate(HORIZONS, 2):
        ratio = row[f'h{h}_ratio']
        stars = row[f'h{h}_stars']
        if np.isnan(ratio):
            c(ws1, r, ci, '—', fill=alt)
            continue
        f = GREEN if ratio < 0.85 else (RED if ratio > 1.15 else alt)
        label = f"{ratio:.3f}{stars}"
        c(ws1, r, ci, label, fill=f, bold=(ratio < 1.0))

    avg = row['avg_ratio']
    f_avg = GREEN if avg < 0.85 else (RED if avg > 1.15 else alt)
    c(ws1, r, 11,
      f"{avg:.3f}" if not np.isnan(avg) else '—',
      fill=f_avg, bold=(avg < 1.0 if not np.isnan(avg) else False))

ws1.freeze_panes = 'A4'

# ── Sheet 2: p-values table ───────────────────────────────────────────────────
ws2 = wb.create_sheet('p-values')
sw(ws2, [38] + [10]*9)

c(ws2, 1, 1,
  'DM Test p-values (two-sided)  |  Bold = significant at 10%  |  '
  'Green = significant at 5%',
  fill=BLUE, bold=True, color='FFFFFF', cs=10)
ws2.row_dimensions[1].height = 25

c(ws2, 2, 1, 'Model', fill=DBLUE, bold=True, color='FFFFFF', align='left')
for ci, h in enumerate(HORIZONS, 2):
    c(ws2, 2, ci, f'h={h}', fill=DBLUE, bold=True, color='FFFFFF')

for ri, (_, row) in enumerate(summary.iterrows()):
    r   = ri + 3
    alt = GREY if ri % 2 == 0 else WHITE
    m   = row['model']
    c(ws2, r, 1, m, fill=alt, align='left')
    for ci, h in enumerate(HORIZONS, 2):
        pval = row[f'h{h}_pval']
        if np.isnan(pval):
            c(ws2, r, ci, '—', fill=alt)
            continue
        f = GREEN if pval <= 0.05 else (LBLUE if pval <= 0.10 else alt)
        c(ws2, r, ci, round(pval, 4), fill=f,
          bold=(pval <= 0.10), fmt='0.0000')

ws2.freeze_panes = 'A3'

# ── Sheet 3: DM statistics ────────────────────────────────────────────────────
ws3 = wb.create_sheet('DM Statistics')
sw(ws3, [38] + [10]*9)

c(ws3, 1, 1,
  'DM Test Statistics  |  Positive = model worse than benchmark  |  '
  'Negative = model better than benchmark',
  fill=BLUE, bold=True, color='FFFFFF', cs=10)
ws3.row_dimensions[1].height = 25

c(ws3, 2, 1, 'Model', fill=DBLUE, bold=True, color='FFFFFF', align='left')
for ci, h in enumerate(HORIZONS, 2):
    c(ws3, 2, ci, f'h={h}', fill=DBLUE, bold=True, color='FFFFFF')

for ri, (_, row) in enumerate(summary.iterrows()):
    r   = ri + 3
    alt = GREY if ri % 2 == 0 else WHITE
    m   = row['model']
    c(ws3, r, 1, m, fill=alt, align='left')
    for ci, h in enumerate(HORIZONS, 2):
        dm = row[f'h{h}_dm']
        if np.isnan(dm):
            c(ws3, r, ci, '—', fill=alt)
            continue
        f = GREEN if dm < -1.645 else (RED if dm > 1.645 else alt)
        c(ws3, r, ci, round(dm, 3), fill=f, fmt='0.000')

ws3.freeze_panes = 'A3'

wb.save(os.path.join(script_dir, OUTPUT_FILE))
print(f"  Saved: {OUTPUT_FILE}")
print()
print("=" * 60)
print("COMPLETE")
print("=" * 60)
print(f"  {OUTPUT_FILE}")
print(f"  Sheets: MSPE Ratios with DM Stars | p-values | DM Statistics")


Loading data...
  Rows: 29,133  |  Models: 27  |  Origins: 131
Running DM tests...
  Done.

DM TEST RESULTS — Two-sided, ** = 5%, * = 10%
Benchmark: RW_avg (benchmark)
Model                                          h= 1  h= 3  h= 6  h= 9  h=12  h=15  h=18  h=21  h=24  Average
--------------------------------------------------------------------------------
AR(12)                                          1.002   0.920   0.733   0.697   0.628  0.580*  0.543*  0.526**  0.523**  0.684
BAR(12)                                         1.003   0.923   0.736   0.698   0.629  0.580*  0.544*  0.526**  0.524**  0.685
VEC(12)                                         0.966   1.044   0.827   0.819   0.801   0.661  0.597*  0.580*  0.577**  0.763
ARMA(1,1)                                       0.956   0.989   0.899   0.841   0.749  0.675*  0.626**  0.596**  0.580**  0.768
PriceSpread(41) α=0,β̂                          1.001   0.915   0.848   0.816   0.781  0.748*  0.737*   0.753   0.768  0.819
PriceSpre